# MOMENT anomaly scoring — DIMER task-inference tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook spec:** `1.0`  
**Capability:** raw reconstruction-residual anomaly ranking  
**Upstream model:** `AutonLab/MOMENT-1-base` at immutable revision `9fea447e740eb968a9e8d80c7562ae122bdb5dde`

This notebook demonstrates **raw reconstruction-residual scoring**, not a binary detector. MOMENT sees each scored point and the repository reports its self-reconstruction residual. Under the default MAE rule, **higher residual scores mean stronger anomaly evidence according to this score**. There is **no universal/default threshold in v1** and this tutorial never converts scores into binary anomaly labels.

**No gradient training, fine-tuning, in-context conditioning, or fitted preprocessing occurs.**

**Upstream vs. this repository.** Upstream MOMENT supplies the pretrained reconstruction model. This repository supplies immutable pinning/integrity verification, long-format validation and canonicalization, explicit residual/aggregation policies, scored-domain accounting, threshold non-policy, and machine-readable provenance.

**By the end of this notebook you will be able to:** verify runtime/model identity; load a deterministic labelled sample or BYOD CSV; validate/canonicalize input; compute raw anomaly scores through the production-facing API; interpret score direction and threshold semantics; check injected-spike ranking quantitatively; and export raw scores with provenance.

**This notebook does not demonstrate:** a calibrated detector, a universal threshold, forecasting, classification, or production fitness. High reconstruction error and real-world anomaly status are not equivalent concepts.

References: [repository README](https://github.com/kurtvalcorza/moment-pipeline), [model card](https://github.com/kurtvalcorza/moment-pipeline/blob/main/MODEL_CARD.md), [sample dataset card](https://github.com/kurtvalcorza/moment-pipeline/blob/main/examples/sample-data/DATASET_CARD.md), [upstream MOMENT](https://github.com/moment-timeseries-foundation-model/moment), and [pinned model repository](https://huggingface.co/AutonLab/MOMENT-1-base).


## Prerequisites and data contract

- **Runtime:** Python 3.12; CPU default; public v1 inference is `float32` only.
- **Network:** first run needs GitHub and Hugging Face access; pinned weights are ~454 MB. No credentials are required for the default public path.
- **Default data:** deterministic synthetic data with three documented injected spikes in the `vibration` channel. Labels make ranking behavior falsifiable; they are not calibration data or benchmark evidence.
- **BYOD schema:** one UTF-8 CSV with `series_id`, `timestamp`, `channel`, `value`. BYOD does not require anomaly labels; without labels, the notebook ranks residuals but cannot measure detector quality. Duplicate or ambiguous column names are rejected from the raw header before dataframe parsing, and duplicate row keys plus invalid/non-finite values are rejected by production validation.
- **Operational ceilings:** 5,000,000 rows, 1,024 series, 32 channels, 1,024 windows; 512-step windows and 8-step patches. Long series keep the final 512 timestamps; short series are left-padded; irregular spacing is surfaced.
- **BYOD privacy:** the pipeline does not transmit uploaded CSV contents to an external inference service. Colab uploads live in Google's hosted runtime; model files are fetched separately. Do not upload confidential, restricted, or sensitive data unless the runtime is authorized.

The default path is non-interactive. There is no stochastic sampling or random split; injected sample positions are deterministic. Numerical values can still vary slightly across hardware/framework implementations, so no cross-platform bitwise reproducibility claim is made.


## 1. Bootstrap the repository and locked runtime

This stage installs the pinned `uv` bootstrap when needed, obtains the repository checkout, and installs the exported lock graph before installing this package without resolving a second dependency graph. Successful completion means the repository checkout is identified and the locked runtime is installed without intentional dependency drift; it does **not** yet verify or execute the model weights.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/moment-pipeline.git"
REPO_NAME = "moment-pipeline"
UV_VERSION = "0.12.9"
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}"],
        check=True,
    )
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    ROOT = Path.cwd()
    subprocess.run(
        ["uv", "pip", "install", "--system", "-r", "requirements.lock.txt"],
        check=True,
    )
    subprocess.run(
        ["uv", "pip", "install", "--system", "--no-deps", "-e", "."],
        check=True,
    )
else:
    print(f"Repository checkout detected: {ROOT}")
repo_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("repository commit:", repo_commit)


## 2. Inspect runtime, model identity, and limits

This stage exposes the effective Python/library versions, device and precision policy, immutable model identity, and resource ceilings **before inference**. Successful output means the visible execution contract is supported; it does not mean the checkpoint has passed integrity verification yet.


In [ ]:
import platform
from importlib.metadata import version

import torch

from moment_pipeline import (
    MomentConfig,
    PINNED_MODEL_ID,
    PINNED_REVISION,
    __version__ as moment_pipeline_version,
)

config = MomentConfig(task="reconstruction", device="cpu")
limits = config.limits
print("Python:", platform.python_version())
print("moment-pipeline:", moment_pipeline_version)
print("momentfm:", version("momentfm"))
print("PyTorch:", torch.__version__)
print("device:", config.resolved_device())
print("dtype:", config.dtype)
print("model:", PINNED_MODEL_ID)
print("immutable revision:", PINNED_REVISION)
print(
    "limits:",
    {
        "max_rows": limits.max_rows,
        "max_series": limits.max_series,
        "max_channels": limits.max_channels,
        "max_windows": limits.max_windows,
        "sequence_length": config.sequence_length,
        "patch_length": config.patch_length,
    },
)


## 3. Load the labelled synthetic demonstration sample or BYOD

The default sample and labels are generated by repository code and verified against `SHA256SUMS`. The optional upload path first validates the **raw CSV header** with `read_long_csv_bytes()` so duplicate or ambiguous names cannot be silently renamed by pandas. When BYOD is enabled no anomaly labels are assumed; the notebook still scores and exports the series but skips labelled ranking metrics. Successful completion means one identified input frame is available and its source/digest are recorded.


In [ ]:
import hashlib
import json

import pandas as pd

from moment_pipeline import read_long_csv_bytes

USE_BYOD = False  # @param {type:"boolean"}
if USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "BYOD upload is available in Colab. Outside Colab, load a CSV into `frame` "
            "with columns series_id,timestamp,channel,value and validate it before inference."
        ) from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV with columns series_id,timestamp,channel,value.")
    name, payload = next(iter(uploaded.items()))
    frame = read_long_csv_bytes(payload)
    labels = None
    sample_identity = {
        "kind": "byod",
        "name": name,
        "sha256": hashlib.sha256(payload).hexdigest(),
    }
    print(f"Loaded BYOD: {name}")
else:
    sample_root = ROOT / "examples" / "sample-data"
    subprocess.run([sys.executable, str(sample_root / "generate_samples.py")], check=True)
    sample_path = sample_root / "moment_anomaly.csv"
    label_path = sample_root / "moment_anomaly_labels.csv"
    manifest = {}
    for line in (sample_root / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        digest, filename = line.split("  ", 1)
        manifest[filename] = digest
    sample_digest = hashlib.sha256(sample_path.read_bytes()).hexdigest()
    label_digest = hashlib.sha256(label_path.read_bytes()).hexdigest()
    assert sample_digest == manifest[sample_path.name], "sample digest mismatch"
    assert label_digest == manifest[label_path.name], "label digest mismatch"
    frame = pd.read_csv(sample_path)
    labels = pd.read_csv(label_path)
    labels["timestamp"] = pd.to_datetime(labels["timestamp"])
    sample_identity = {
        "kind": "synthetic",
        "name": sample_path.name,
        "sha256": sample_digest,
        "labels_name": label_path.name,
        "labels_sha256": label_digest,
    }
    print(
        f"Loaded verified synthetic sample rows={len(frame)}, "
        f"injected anomalies={int(labels['is_injected_anomaly'].sum())}"
    )
print(frame.head())
print("sample identity:", sample_identity)


## 4. Validate and canonicalize

This stage runs the repository's production-facing schema/value/resource validation and converts the input to canonical 512-step windows. Successful output means the data contract passed and the summary makes padding, truncation, source missingness, and irregular-frequency handling visible before the model runs.


In [ ]:
from moment_pipeline import to_windows, validate_long_frame

report, normalized = validate_long_frame(frame, config)
windows = to_windows(normalized, config, report=report, frame=normalized)
print(
    "validation:",
    {
        "rows": report.n_rows,
        "series": len(report.series_ids),
        "channels": len(report.channels),
        "irregular_series": list(report.irregular_series),
    },
)
print("window tensor:", windows.x_enc.shape)
print("channels:", windows.channels)
print("padded windows:", int(sum(windows.padded)), "/", windows.n_windows)
print("truncated windows:", int(sum(windows.truncated)), "/", windows.n_windows)
print("source missing fraction:", windows.masked_point_fraction)
if any(windows.truncated):
    print("WARNING: long input series were truncated to their final 512 timestamps.")
if any(windows.padded):
    print("NOTE: short input series were left-padded; padding is excluded from scoring.")


## 5. Resolve the pinned checkpoint and compute raw anomaly scores

`load_moment()` integrity-verifies the immutable reconstruction checkpoint. The core operation then uses `score_anomalies(..., loss="mae", channel_aggregation="none")`; `anomaly_score` is an uncalibrated absolute reconstruction residual per scored series/channel/timestamp. **Higher = larger reconstruction discrepancy.** The pipeline deliberately ships no binary decision threshold. Successful output means this validated input was scored with the verified model under the displayed score/threshold policy; it does not mean those scores are calibrated anomaly probabilities.


In [ ]:
from moment_pipeline import build_provenance, load_moment, score_anomalies

model = load_moment(task="reconstruction", device="cpu")
result = score_anomalies(
    windows,
    model,
    loss="mae",
    channel_aggregation="none",
    warmup=False,
)
provenance = build_provenance(model, windows, result)
scores = result.to_frame()
print("effective model:", model.identity.name)
print("effective revision:", model.identity.revision)
print("verified weight file:", model.identity.weight_file_loaded)
print("score policy:", result.score_policy)
print("threshold policy:", result.threshold_policy)
print("scored fraction:", result.scored_point_fraction)


## 6. Rank residuals and check the injected demonstration points

For the bundled labelled sample, this stage ranks `vibration` residuals from highest to lowest. **Top-k recall** uses `k` equal to the number of injected spikes and asks how many injected points appear among the same number of highest-scoring positions. Successful output is a falsifiable tutorial ranking check, **not** a calibrated detector metric or upstream benchmark. For BYOD without labels, only the highest residuals are displayed and no correctness claim is made.


In [ ]:
score_channel = (
    "vibration" if "vibration" in set(scores["channel"]) else str(scores["channel"].iloc[0])
)
ranked = (
    scores[(scores["channel"] == score_channel) & scores["scored"]]
    .copy()
    .sort_values("anomaly_score", ascending=False)
    .reset_index(drop=True)
)
if ranked.empty:
    raise ValueError(
        "No scored positions are available after masking/padding; provide a series with observed values."
    )
ranked["rank"] = ranked.index + 1
if labels is not None:
    ranked = ranked.merge(labels, on=["series_id", "timestamp"], how="left")
    ranked["is_injected_anomaly"] = (
        ranked["is_injected_anomaly"].fillna(False).astype(bool)
    )
    n_injected = int(ranked["is_injected_anomaly"].sum())
    top_k = ranked.head(n_injected)
    top_k_recall = (
        float(top_k["is_injected_anomaly"].sum()) / n_injected
        if n_injected
        else float("nan")
    )
    print(
        ranked[["rank", "timestamp", "anomaly_score", "is_injected_anomaly"]].head(12)
    )
    print(
        "injected ranks:",
        ranked.loc[
            ranked["is_injected_anomaly"],
            ["timestamp", "rank", "anomaly_score"],
        ].to_dict("records"),
    )
    print(f"tutorial top-{n_injected} recall:", top_k_recall)
else:
    top_k_recall = None
    print(ranked[["rank", "series_id", "timestamp", "anomaly_score"]].head(12))
    print("No labels supplied: ranking shown without a correctness metric.")


## 7. Visualize raw score over time

The diagnostic plot shows the raw residual and adds no decision threshold. Smooth drift or other real anomalies that the reconstruction model reproduces well may score low, while benign but hard-to-reconstruct events may score high. Successful rendering makes score behavior easier to inspect; it does not replace the machine-readable score table or establish detector calibration.


In [ ]:
def write_line_svg(path, layers, *, title, width=760, height=280):
    all_values = [float(value) for _, values in layers for value in values]
    low, high = min(all_values), max(all_values)
    span = high - low or 1.0
    max_points = max(len(values) for _, values in layers)
    left, right, top, bottom = 48, width - 20, 30, height - 38

    def point(index, value):
        x = left + (right - left) * index / max(max_points - 1, 1)
        y = bottom - (bottom - top) * (float(value) - low) / span
        return f"{x:.1f},{y:.1f}"

    svg = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
        f'<text x="{left}" y="18" font-family="sans-serif" font-size="14">{title}</text>',
    ]
    for idx, (label, values) in enumerate(layers):
        points = " ".join(point(i, value) for i, value in enumerate(values))
        stroke = ["#111827", "#2563eb", "#dc2626"][idx % 3]
        svg.append(
            f'<polyline fill="none" stroke="{stroke}" stroke-width="2" points="{points}"/>'
        )
        svg.append(
            f'<text x="{left + 180 * idx}" y="{height - 10}" font-family="sans-serif" '
            f'font-size="12" fill="{stroke}">{label}</text>'
        )
    svg.append("</svg>")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(svg), encoding="utf-8")
    return path

ordered = ranked.sort_values("timestamp")
score_plot = write_line_svg(
    ROOT / "outputs" / "moment_anomaly_scores.svg",
    [("raw MAE residual", ordered["anomaly_score"].fillna(0.0).tolist())],
    title=f"MOMENT raw anomaly score — {score_channel}",
)
try:
    from IPython.display import SVG, display

    display(SVG(filename=str(score_plot)))
except ImportError:
    print(f"SVG written to {score_plot}")


## 8. Export raw scores and provenance

This stage writes `outputs/moment_anomaly_scores.csv` and `outputs/moment_anomaly_provenance.json` (plus the diagnostic SVG). Successful completion establishes that identifier-preserving residual scores, their score/evaluation policy, and model/runtime/data provenance are serialized under stable filenames. Export success does **not** establish that residuals are calibrated anomaly decisions or that the bundled ranking result generalizes.


In [ ]:
output_dir = ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(output_dir / "moment_anomaly_scores.csv", index=False)
provenance["data"] = sample_identity
provenance["evaluation"] = {
    "estimation_procedure": (
        "deterministic injected-spike ranking on bundled synthetic sample"
        if labels is not None
        else "unlabelled BYOD residual ranking; no correctness metric"
    ),
    "sample_evidence_only": True,
    "top_k_recall": top_k_recall,
}
(output_dir / "moment_anomaly_provenance.json").write_text(
    json.dumps(provenance, indent=2, default=str),
    encoding="utf-8",
)
print("exports:", sorted(path.name for path in output_dir.glob("moment_anomaly*")))


## Interpretation, limits, and next steps

A successful run proves that the repository can validate this input, resolve and integrity-check the pinned MOMENT reconstruction checkpoint, compute the documented raw residual score over its valid scored domain, rank those scores, and export identifier-preserving outputs with provenance. On the bundled sample it also provides a falsifiable check of how the three injected spikes rank.

It **does not prove** that high residuals are real-world anomalies, that low residuals are normal, that the bundled top-k result generalizes, or that any numerical threshold is calibrated. The synthetic labels are demonstration evidence only. Deployment thresholds, if needed, belong to the downstream application and require representative calibration/validation data and an explicit false-positive/false-negative cost model.

A useful next experiment is to build a domain-specific labelled validation set containing both true anomalies and difficult normal events, then evaluate ranking and calibration separately.
